In [ ]:
# Purpose - makes fields/spectra movie (newer)

findex = 110 #6 #25

In [ ]:
# Imports and Folder Selection

from graphet import Data
from graphet.plugins import TristanV2 
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import sys; sys.path.append('..')
from useful_funcs import *
sys.path.append('../fmgmt')
from TristanRun import TristanRun

%matplotlib widget

plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 12

# Read Input File Data

run = TristanRun(findex)
gyrofreq = run.gyrofreq
boilerplate = run.boilerplate + f"tstep = {run.steps[0]}"
cc = run.cc
B0 = run.B0

my_steps = run.steps #[run.steps[0]]

In [ ]:
# Configure Data

d = run.config_data(my_steps, "fields")

In [ ]:
def fq_cut(fields, timestep, rect):
    xval1 = rect[0]
    xval2 = rect[1]

    yval1 = rect[2]
    yval2 = rect[3]

    x1, x2 = idcs(xval1, xval2, fields.x.values)
    y1, y2 = idcs(yval1, yval2, fields.y.values)

    return fields.isel(t = timestep).isel(z = 0).isel(x = slice(x1, x2 + 2)).isel(y = slice(y1, y2 + 2))

In [ ]:
def plot_fq3(plot, field_data, quantity, cbar_label, range, cmap):
    # Plots using imshow, slower than pcolormesh but shows the value upon mouseover

    my_extent = [min(field_data.x), max(field_data.x), min(field_data.y), max(field_data.y)]
    #viridis

    if (range is None):
        mesh = plot.imshow(quantity, cmap = cmap, origin = 'lower', extent = my_extent)
    else:
        mesh = plot.imshow(quantity, cmap = cmap, vmax = range[1], vmin = range[0], origin = 'lower', extent = my_extent)
    
    plot.set_xlabel('$x$ [$c/\omega_{p}$]'); plot.set_ylabel('$y$ [$c/\omega_{p}$]')

    cbar = plot.figure.colorbar(mesh, ax=plot)

    plot.set_aspect('equal')

In [ ]:

# Plot Domain and Spectrum

counter = 0

for abs_ts in my_steps[-1:]:
    plt.close()
    fig, (ax1, ax2) = plt.subplots(nrows = 1, ncols = 2, figsize = (10, 8))

    rect = [0, run.xlim, 0, run.ylim]

    #rects = {
    #    2: [45, 75, 80, 170],
    #    6: [45, 75, 80, 170]
    #}

    #rect = rects[findex]

    roi = fq_cut(d.fields, abs_ts - my_steps[0], rect)

    cs = roi

    #ez_treal = cs.bx * cs.vely - cs.by * cs.velx 
    #ez_nontreal = cs.ez - ez_treal 
    #ez_tot = cs.ez
    #JdotE = cs.ex * cs.jx + cs.ey * cs.jy + cs.ez * cs.jz

    dens = cs.dens1 + cs.dens2

    esq, bsq, chi = sqrd(cs) 

    v_A = run.beta_A

    pows = {
        0.3: 0.35,
        10: 0.25
    }

    plot_basic(ax1, run, roi, sdens = 0.5)

    plt.suptitle(run.boilerplate + f"\n $t \Omega$ =  {gyrofreq * abs_ts:.2f} ({abs_ts})")

    ax1.set_xlim(rect[:2])
    ax1.set_ylim(rect[2:])

    species = 1
    bs = run.find_total_spectrum(abs_ts, species)

    ax2.loglog(bs.ebins, bs.calc_hist(), linewidth = 1, markersize = 1.5)
    #ax2.set_title("Energy Spectrum")
    ax2.grid(visible = True)
    ax2.set_xlim([1e-4, 1e2])
    #ax2.set_ylim([1, 1e12])
    #ax2.loglog(spec_bins, spec_counts, linewidth = 1, markersize = 1.5)

    plt.xlabel("$(\gamma - 1) / \sigma$")    
    plt.ylabel("$dN / d\gamma$")

    plt.show()

    plt.savefig(f"/scratch/gpfs/ar3676/analysis/kino/{counter:05d}", dpi = 200)
    counter +=1


In [ ]:
# Plot Domain and charge density

print(0 / 0)

counter = 0

for abs_ts in my_steps[::1]:
    plt.close()
    fig, (ax1, ax2) = plt.subplots(nrows = 1, ncols = 2, figsize = (10, 8))

    dx = 20
    dy = 30

    rect = [run.xlim / 2 - dx, run.xlim / 2 + dx, run.ylim / 2 - dy, run.ylim / 2 + dy]

    roi = fq_cut(d.fields, abs_ts - my_steps[0], rect)

    cs = roi
    dens = cs.dens1 + cs.dens2

    esq, bsq, chi = sqrd(cs) 

    v_A = run.beta_A

    #plt.contour(cs.x, cs.y, chi, [0], colors = 'white')
    #plt.contour(cs.x, cs.y, chi_adj, [0], colors = 'white')
    plot_streamline(ax1, cs, cs.bx, cs.by, 0.5, 'white') 
    plot_fq3(ax1, cs, (cs.dens1 + cs.dens2) / run.ppc, "$n / n_0$", [0, 5], 'viridis')

    ax1.set_title("Particle Density ($n / n_0$)")
    
    plt.suptitle(run.boilerplate + f"\n $t \Omega$ =  {gyrofreq * abs_ts:.2f} ({abs_ts})")

    #ax1.set_xlim(rect[:2])
    #ax1.set_ylim(rect[2:])

    plot_streamline(ax2, cs, cs.bx, cs.by, 0.5, 'black') 

    plot_fq3(ax2, cs, (cs.dens2 - cs.dens1) / run.ppc, r"$\rho_e / (e n_0)$", [-0.5, 0.5], 'seismic')

    ax2.set_title(r"Charge Density$\rho / (e n_0)$")

    ax2.set_xlim(rect[:2])
    ax2.set_ylim(rect[2:])

    plt.suptitle(run.boilerplate + f"\n $t \Omega$ =  {gyrofreq * abs_ts:.2f} ({abs_ts})")

    plt.tight_layout()

    plt.savefig(f"/scratch/gpfs/ar3676/analysis/kino/{counter:05d}", dpi = 200)
    counter +=1
